# K-Means — Segmentacion avanzada de miembros de gimnasio

---

**Autor:** Borja Mora Méndez
**Contacto:** [borja.mora.mendez@gmail.com](mailto:borja.mora.mendez@gmail.com) · [LinkedIn](https://www.linkedin.com/in/borja-mora-mendez/)
**Repositorio:** [Data Analytics Portfolio](https://github.com/BORJAMOME/Data-Analytics-Portfolio)
**Categoría:** Machine Learning · No Supervisado · Clustering · K-Means

---

**Objetivo:** Aplicar K-Means con 4 variables sobre datos reales de un gimnasio, comparar estabilidad de clusters con distintos random_state, y contrastar segmentos con la variable objetivo Abandono.

**Contexto de negocio:** Un gimnasio quiere identificar perfiles de riesgo de abandono para lanzar campanas de retencion antes de que sea demasiado tarde. La hipotesis: los patrones de uso revelan quienes estan a punto de irse.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, silhouette_samples
import os

sns.set_style("whitegrid")

# Estilo visual — sistema de color validado (consejo UX/UI Data)
BACKGROUND    = '#fbfbfb'
PURPLE        = '#7a7bff'   # único color de énfasis / serie única en scatter y líneas (1 por gráfico)
PURPLE_LIGHT  = '#9b9cff'   # EDA de una sola serie (histogramas independientes)
POSITIVE      = '#6b8158'   # exclusivo signo positivo
NEGATIVE      = '#c34031'   # exclusivo signo negativo
NEUTRAL_BAR   = '#d9d9d9'   # barras/áreas de contexto (siempre con etiqueta de valor)
NEUTRAL_LINE  = '#8f8c9e'   # líneas de contexto
CONTEXT_LINES = [NEUTRAL_LINE, '#a89a8a', '#7d94a8']   # gama fija para 2+ líneas de contexto
INK           = '#111111'
MUTED         = '#707070'

# Paleta categórica para identidad de cluster — validada (ΔE OKLab, simulación CVD) para
# pares adyacentes (barras, líneas, enlaces de dendrograma). En scatter/PCA con 4+ clusters
# el color por sí solo no basta para daltonismo severo: por eso cada cluster lleva también
# una forma de marcador distinta (CLUSTER_MARKERS) — nunca dependas solo del color.
CLUSTER_PALETTE = ['#7a7bff', '#eb6834', '#1baf7a', '#e34948', '#eda100', '#e87ba4', '#008300']
CLUSTER_MARKERS = ['o', 's', '^', 'D', 'v', 'P', 'X']

DIVERGING_CMAP = LinearSegmentedColormap.from_list(
    "borja_diverging", ["#c34031", "#e0a89f", "#f0ede8", "#b7c2a9", "#6b8158"]
)
SEQUENTIAL_GREEN = LinearSegmentedColormap.from_list(
    "borja_sequential", [BACKGROUND, POSITIVE]
)

def color_annotations(ax, values, threshold, dark="#ffffff", light=INK):
    """Recolorea el texto de un heatmap celda a celda según su magnitud."""
    for text, value in zip(ax.texts, np.asarray(values).flatten()):
        text.set_color(dark if abs(value) >= threshold else light)

plt.rcParams.update({
    'figure.figsize': (10, 5),
    'figure.dpi': 100,
    'figure.facecolor': BACKGROUND,
    'axes.facecolor': BACKGROUND,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.edgecolor': MUTED,
    'axes.labelcolor': INK,
    'axes.titlesize': 13,
    'axes.titleweight': 'bold',
    'axes.titlecolor': INK,
    'xtick.color': MUTED,
    'ytick.color': MUTED,
    'font.family': 'sans-serif',
    'font.size': 10,
    'grid.color': '#f0f0f0',
    'grid.linewidth': 0.5,
})

print("Librerias cargadas correctamente")

## 1. Carga y exploracion del dataset

In [2]:
ruta = os.path.join(os.path.dirname(os.path.abspath("__file__")),
                    "gym_clientes.xlsx")
df = pd.read_excel(ruta)
print(f"Dataset: {df.shape[0]} registros, {df.shape[1]} variables")
print(f"Columnas: {list(df.columns)}")
display(df.describe())

Dataset: 300 registros, 7 variables
Columnas: ['ID_Cliente', 'Antiguedad_Meses', 'Asistencias_Mes', 'Horas_Pico_Mes', 'Gasto_Mensual_Extra', 'Satisfecho', 'Abandono']


,ID_Cliente,Antiguedad_Meses,Asistencias_Mes,Horas_Pico_Mes,Gasto_Mensual_Extra,Satisfecho,Abandono
count,300.000000,300.000000,300.000000,300.000000,300.000000,300.000000,300.000000
mean,150.500000,18.880000,12.160000,14.669690,81.688590,0.480000,0.163333
std,86.746758,10.603161,7.267539,6.093048,30.467821,0.500435,0.370287
min,1.000000,1.000000,0.000000,1.406971,10.000000,0.000000,0.000000
25%,75.750000,9.000000,6.000000,9.648578,61.246282,0.000000,0.000000
50%,150.500000,20.000000,12.000000,15.072785,82.238850,0.000000,0.000000
75%,225.250000,28.000000,18.250000,19.750393,104.110393,1.000000,0.000000
max,300.000000,35.000000,24.000000,24.000000,145.911455,1.000000,1.000000


## 2. Seleccion de variables y preprocesamiento

Usamos 4 variables de comportamiento. Excluimos Satisfecho y Abandono (son target, no features para clustering).

In [3]:
features = ["Antiguedad_Meses", "Asistencias_Mes", "Horas_Pico_Mes", "Gasto_Mensual_Extra"]
X = df[features].copy()
print(f"Variables seleccionadas: {features}")
print(f"Valores nulos: {X.isnull().sum().sum()}")

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print("Datos estandarizados con StandardScaler")

Variables seleccionadas: ['Antiguedad_Meses', 'Asistencias_Mes', 'Horas_Pico_Mes', 'Gasto_Mensual_Extra']
Valores nulos: 0
Datos estandarizados con StandardScaler


## 3. Metodo del codo y silhouette score

In [ ]:
K_range = range(2, 9)
inertias = []
silhouettes = []

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_scaled, km.labels_))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(K_range, inertias, marker="o", linewidth=2, color=PURPLE)
axes[0].set_xlabel("k")
axes[0].set_ylabel("Inercia")
axes[0].set_title("Metodo del codo")

axes[1].plot(K_range, silhouettes, marker="s", linewidth=2, color=PURPLE)
axes[1].set_xlabel("k")
axes[1].set_ylabel("Silhouette Score")
axes[1].set_title("Silhouette Score por k")

best_k = list(K_range)[np.argmax(silhouettes)]
print(f"Mejor silhouette: k={best_k} ({max(silhouettes):.4f})")

plt.tight_layout()
plt.show()

## 4. Estabilidad de clusters: variacion de random_state

K-Means depende de la inicializacion. Verificamos que los clusters son consistentes ejecutando con 10 semillas diferentes.

In [5]:
k_final = 3
seeds = [0, 7, 13, 21, 42, 55, 77, 99, 123, 256]
scores = []

for s in seeds:
    km = KMeans(n_clusters=k_final, random_state=s, n_init=10)
    labels = km.fit_predict(X_scaled)
    sc = silhouette_score(X_scaled, labels)
    scores.append(sc)

print(f"Silhouette scores con k={k_final} y 10 semillas:")
for s, sc in zip(seeds, scores):
    print(f"  seed={s:>3d} -> {sc:.4f}")

print(f"\nMedia: {np.mean(scores):.4f} | Std: {np.std(scores):.4f}")
print(f"Variacion: {'ESTABLE' if np.std(scores) < 0.01 else 'INESTABLE'}")

Silhouette scores con k=3 y 10 semillas:
  seed=  0 -> 0.3728
  seed=  7 -> 0.3723
  seed= 13 -> 0.3723
  seed= 21 -> 0.3728
  seed= 42 -> 0.3723
  seed= 55 -> 0.3723
  seed= 77 -> 0.3723
  seed= 99 -> 0.3723
  seed=123 -> 0.3723
  seed=256 -> 0.3728

Media: 0.3725 | Std: 0.0002
Variacion: ESTABLE


## 5. Silhouette plot detallado

In [ ]:
kmeans = KMeans(n_clusters=k_final, random_state=42, n_init=10)
labels = kmeans.fit_predict(X_scaled)
sil_avg = silhouette_score(X_scaled, labels)
sil_vals = silhouette_samples(X_scaled, labels)

fig, ax = plt.subplots(figsize=(10, 7))
y_lower = 10

for cl in range(k_final):
    cl_sil = sil_vals[labels == cl]
    cl_sil.sort()
    size = len(cl_sil)
    y_upper = y_lower + size
    ax.fill_betweenx(np.arange(y_lower, y_upper), 0, cl_sil,
                     alpha=0.7, color=CLUSTER_PALETTE[cl % len(CLUSTER_PALETTE)], label=f"Cluster {cl} (n={size})")
    y_lower = y_upper + 10

ax.axvline(x=sil_avg, color=INK, linestyle="--", label=f"Media: {sil_avg:.3f}")
ax.set_xlabel("Silhouette coefficient", fontsize=12)
ax.set_ylabel("Cluster")
ax.set_title("Silhouette plot por cluster", fontsize=14, fontweight="bold")
ax.legend()
plt.tight_layout()
plt.show()

## 6. Perfil de clusters y cross-check con Abandono

In [7]:
df["Cluster"] = labels

perfil = df.groupby("Cluster")[features].mean().round(1)
print("Perfil medio de cada cluster:")
display(perfil)

# Cross-check con variables target
print("\nTasa de Abandono por cluster:")
abandono = df.groupby("Cluster")["Abandono"].mean().round(3) * 100
for cl, tasa in abandono.items():
    print(f"  Cluster {cl}: {tasa:.1f}% abandono")

print("\nSatisfaccion media por cluster:")
satisf = df.groupby("Cluster")["Satisfecho"].mean().round(3) * 100
for cl, tasa in satisf.items():
    print(f"  Cluster {cl}: {tasa:.1f}% satisfechos")

Perfil medio de cada cluster:


,Antiguedad_Meses,Asistencias_Mes,Horas_Pico_Mes,Gasto_Mensual_Extra
Cluster,,,,
0,19.8,4.6,8.4,54.6
1,26.8,17.6,19.4,113.2
2,8.8,16.7,18.3,84.7



Tasa de Abandono por cluster:
  Cluster 0: 40.8% abandono
  Cluster 1: 0.0% abandono
  Cluster 2: 0.0% abandono

Satisfaccion media por cluster:
  Cluster 0: 7.5% satisfechos
  Cluster 1: 80.0% satisfechos
  Cluster 2: 69.4% satisfechos


## 7. Visualizacion: boxplots por cluster

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for i, feat in enumerate(features):
    ax = axes[i // 2, i % 2]
    data_per_cluster = [df[df["Cluster"] == cl][feat].values for cl in range(k_final)]
    bp = ax.boxplot(data_per_cluster, patch_artist=True, labels=[f"C{cl}" for cl in range(k_final)])
    for j, box in enumerate(bp["boxes"]):
        box.set_facecolor(CLUSTER_PALETTE[j % len(CLUSTER_PALETTE)])
        box.set_alpha(0.7)
    ax.set_title(feat, fontsize=13, fontweight="bold")
    ax.set_xlabel("Cluster")

fig.suptitle("Distribucion de variables por cluster", fontsize=15, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

## 8. Conclusiones y recomendaciones

**Hallazgos:**
- K-Means con k=3 produce clusters estables (baja varianza entre semillas).
- Los clusters capturan perfiles de uso diferenciados que correlacionan con la tasa de abandono.
- El silhouette plot muestra la cohesion interna de cada segmento.

**Recomendaciones para el gimnasio:**
- **Cluster de alto riesgo:** Campana de retencion proactiva (descuentos, entrenador personal gratis).
- **Cluster fidelizado:** Programa de referidos — estos miembros ya estan satisfechos.
- **Cluster intermedio:** Monitorizar y activar si la asistencia cae 2 semanas seguidas.

**Diferencia vs. clustering jerarquico:**
- K-Means permite analizar estabilidad via multiples inicializaciones — algo que el jerarquico no ofrece.
- Los centroides de K-Means facilitan la asignacion de nuevos clientes en tiempo real (produccion).